# Module 5 - A Neural Language Model

## What you'll build

The bigram model only remembers one character. Now we give the model a
wider **context window** and let it *learn* (not just count) how to
predict the next character. This is the classic Bengio-style MLP
language model, written in plain NumPy with backprop derived by hand.

Two pieces to implement:

1. **`softmax(logits)`** -- turns raw scores into probabilities. The
   trick that matters: subtract the per-row max before exponentiating,
   so large logits don't overflow to infinity. Mathematically identical,
   numerically safe.
2. **`MLPLM`** -- the model itself:
   * an **embedding** table `C` that maps each token id to a small dense
     vector,
   * a hidden layer `W1, b1` with a `tanh` nonlinearity,
   * an output layer `W2, b2` producing one logit per vocabulary token.
   `forward` flattens the embedded context and runs it through the two
   layers. `loss` is mean cross-entropy. `step` does one SGD update with
   **hand-derived gradients** for every parameter -- read the comments
   in the reference to see each gradient spelled out.

Train it on a tiny batch and the loss should drop sharply: the model is
memorising the mapping, which proves the gradients are correct.

In [ ]:
import numpy as np

def softmax(logits):
    # TODO: numerically stable row-wise softmax. Subtract the per-row max,
    #       exponentiate, divide by the row sum.
    raise NotImplementedError

class MLPLM:
    def __init__(self, vocab, n_embd, block, hidden, seed):
        # TODO: store dims; with np.random.default_rng(seed) init
        #       C (vocab x n_embd), W1 (block*n_embd x hidden), b1,
        #       W2 (hidden x vocab), b2. See the reference for the scales.
        raise NotImplementedError

    def forward(self, X):
        # TODO: embed X (B, block) -> (B, block, n_embd), flatten,
        #       tanh hidden layer, then linear to (B, vocab) logits.
        raise NotImplementedError

    def loss(self, X, Y):
        # TODO: mean cross-entropy of the next-token predictions.
        raise NotImplementedError

    def step(self, X, Y, lr):
        # TODO: one SGD update with hand-derived gradients for C, W1, b1,
        #       W2, b2.
        raise NotImplementedError


## Reveal the reference solution

Run the hidden cell to load the reference `softmax` and `MLPLM` (it
redefines both with the complete implementation).

In [ ]:
# Reference solution (single source of truth: reference/mlp_lm/model.py)

"""A small neural char-level language model (Bengio-style MLP).

This uses plain NumPy arrays for both the forward pass and the gradients.
Backprop is written by hand (no autograd engine) so students can see exactly
how each gradient is derived.
"""

import numpy as np


def softmax(logits):
    """Row-wise softmax that is numerically stable.

    Subtracting the per-row max keeps exponentials from overflowing while
    leaving the result unchanged. Works on 2D arrays; each row sums to 1.
    """
    shifted = logits - logits.max(axis=-1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=-1, keepdims=True)


class MLPLM:
    """An MLP that predicts the next token from a fixed context window.

    Architecture: embed each of ``block`` context positions into ``n_embd``
    dimensions, concatenate them, pass through one tanh hidden layer, then a
    linear layer to per-token logits.
    """

    def __init__(self, vocab, n_embd, block, hidden, seed):
        self.vocab = vocab
        self.n_embd = n_embd
        self.block = block
        self.hidden = hidden

        rng = np.random.default_rng(seed)
        fan_in = block * n_embd
        # Small random init keeps the initial tanh away from saturation.
        self.C = rng.standard_normal((vocab, n_embd)) * 0.1
        self.W1 = rng.standard_normal((fan_in, hidden)) * (1.0 / np.sqrt(fan_in))
        self.b1 = np.zeros(hidden)
        self.W2 = rng.standard_normal((hidden, vocab)) * (1.0 / np.sqrt(hidden))
        self.b2 = np.zeros(vocab)

    def forward(self, X):
        """Map an (B, block) int context array to (B, vocab) logits."""
        emb = self.C[X]                                  # (B, block, n_embd)
        flat = emb.reshape(emb.shape[0], -1)             # (B, block*n_embd)
        h = np.tanh(flat @ self.W1 + self.b1)            # (B, hidden)
        logits = h @ self.W2 + self.b2                   # (B, vocab)
        return logits

    def loss(self, X, Y):
        """Mean cross-entropy of the next-token predictions."""
        logits = self.forward(X)
        probs = softmax(logits)
        B = X.shape[0]
        # Pick the predicted probability of each true target, then -mean(log).
        correct = probs[np.arange(B), Y]
        return float(-np.log(correct).mean())

    def step(self, X, Y, lr):
        """One SGD update with hand-derived gradients for every parameter."""
        B = X.shape[0]

        # ---- forward (cache intermediates needed for backprop) ----
        emb = self.C[X]                                  # (B, block, n_embd)
        flat = emb.reshape(B, -1)                        # (B, block*n_embd)
        h_pre = flat @ self.W1 + self.b1                 # (B, hidden)
        h = np.tanh(h_pre)                               # (B, hidden)
        logits = h @ self.W2 + self.b2                   # (B, vocab)
        probs = softmax(logits)                          # (B, vocab)

        # ---- backward ----
        # dL/dlogits for mean cross-entropy: (softmax - onehot) / B
        dlogits = probs.copy()
        dlogits[np.arange(B), Y] -= 1.0
        dlogits /= B

        dW2 = h.T @ dlogits                              # (hidden, vocab)
        db2 = dlogits.sum(axis=0)                        # (vocab,)

        dh = dlogits @ self.W2.T                         # (B, hidden)
        dh_pre = dh * (1.0 - h ** 2)                     # tanh'(x) = 1 - tanh(x)^2

        dW1 = flat.T @ dh_pre                            # (block*n_embd, hidden)
        db1 = dh_pre.sum(axis=0)                         # (hidden,)

        dflat = dh_pre @ self.W1.T                       # (B, block*n_embd)
        demb = dflat.reshape(emb.shape)                  # (B, block, n_embd)

        # Scatter the embedding gradients back into rows of C.
        dC = np.zeros_like(self.C)
        np.add.at(dC, X, demb)

        # ---- SGD update ----
        self.C -= lr * dC
        self.W1 -= lr * dW1
        self.b1 -= lr * db1
        self.W2 -= lr * dW2
        self.b2 -= lr * db2


## Check your work

The grader checks the forward shape, that softmax stays finite on huge
logits, that the loss is finite and non-negative, and -- the real test
-- that training on a tiny batch overfits it (loss drops below half).

In [ ]:
import sys, os
# Make the repo root importable so `grader` / `reference` resolve.
_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)
from grader.checks.m5 import check_mlp_lm
from grader.core import run_checks

r = run_checks(check_mlp_lm(softmax, MLPLM))
print('PASS' if r.passed else 'FAIL', f'score={r.score:.2f}')
if not r.passed:
    print('failed checks:', r.failed_checks)
